In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import pandas as pd

train = pd.read_csv("train.csv")
dev = pd.read_csv("dev.csv")
test = pd.read_csv("test.csv")

print("===== 데이터 크기 =====")
print("Train:", train.shape)
print("Dev:", dev.shape)
print("Test:", test.shape)

print("\n===== 컬럼 =====")
print("Train:", train.columns.tolist())
print("Dev:", dev.columns.tolist())
print("Test:", test.columns.tolist())

print("\n===== 결측치 =====")
print("Train:", train.isnull().sum())
print("\nDev:", dev.isnull().sum())
print("\nTest:", test.isnull().sum())

print("\n===== Train 정답 분포 =====")
print(train["answer"].value_counts().sort_index())
print(train["answer"].value_counts(normalize=True).sort_index().round(4))

In [ ]:
import pandas as pd

answer_cols = [
    "answer1",
    "answer2",
    "answer3",
    "answer4",
    "answer5"
]

# 응답값 형식 통일
for col in answer_cols:
    dev[col] = (
        dev[col]
        .astype("string")
        .str.strip()
        .str.lower()
    )

# a~d 외의 값은 결측치로 처리
dev[answer_cols] = dev[answer_cols].where(
    dev[answer_cols].isin(["a", "b", "c", "d"])
)

def analyze_answers(row):
    answers = row[answer_cols].dropna()

    if len(answers) == 0:
        return pd.Series([None, 0, 0, 0.0, False])

    counts = answers.value_counts()
    max_count = int(counts.max())

    # 최다 응답이 여러 개인지 확인
    winners = counts[counts == max_count].index.tolist()
    is_tie = len(winners) > 1

    majority_answer = None if is_tie else winners[0]
    agreement_ratio = max_count / len(answers)

    return pd.Series([
        majority_answer,
        len(answers),
        max_count,
        agreement_ratio,
        is_tie
    ])

dev[
    [
        "pseudo_answer",
        "response_count",
        "agreement_count",
        "agreement_ratio",
        "is_tie"
    ]
] = dev.apply(analyze_answers, axis=1)

print("===== 실제 응답자 수 =====")
print(dev["response_count"].value_counts().sort_index())

print("\n===== 최다 응답 일치 인원 =====")
print(dev["agreement_count"].value_counts().sort_index())

print("\n===== 동률 데이터 =====")
print(dev["is_tie"].value_counts())

print("\n===== 높은 신뢰도 데이터 =====")
print("5개 응답 모두 일치:", len(dev[
    (dev["response_count"] == 5) &
    (dev["agreement_count"] == 5)
]))

print("최소 4개 응답, 80% 이상 일치:", len(dev[
    (dev["response_count"] >= 4) &
    (dev["agreement_ratio"] >= 0.8) &
    (~dev["is_tie"])
]))

print("최소 3개 응답, 100% 일치:", len(dev[
    (dev["response_count"] >= 3) &
    (dev["agreement_ratio"] == 1.0)
]))

In [ ]:
high_conf_dev = dev[
    (dev["agreement_count"] == 3) &
    (~dev["is_tie"]) &
    (dev["pseudo_answer"].notna())
].copy()

print("고신뢰 Dev:", len(high_conf_dev))
print("\nPseudo-label 분포:")
print(high_conf_dev["pseudo_answer"].value_counts().sort_index())
print(
    high_conf_dev["pseudo_answer"]
    .value_counts(normalize=True)
    .sort_index()
    .round(4)
)

In [ ]:
pseudo_dev = high_conf_dev[
    ["id", "path", "question", "a", "b", "c", "d"]
].copy()

pseudo_dev["answer"] = high_conf_dev["pseudo_answer"]

print(pseudo_dev.shape)
display(pseudo_dev.head())

In [ ]:
print(
    high_conf_dev
    .groupby(["response_count", "agreement_count"])
    .size()
)

In [ ]:
# 3명 중 3명 또는 4명 중 3명 일치
dev_strong = high_conf_dev[
    high_conf_dev["response_count"].isin([3, 4])
].copy()

# 5명 중 3명 일치
dev_normal = high_conf_dev[
    high_conf_dev["response_count"] == 5
].copy()

print("강한 신뢰:", len(dev_strong))
print("일반 신뢰:", len(dev_normal))

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Train과 동일한 형식으로 변환
train_columns = [
    "id", "path", "question",
    "a", "b", "c", "d", "answer"
]

def make_train_format(df):
    result = df[
        ["id", "path", "question", "a", "b", "c", "d"]
    ].copy()

    result["answer"] = df["pseudo_answer"].values

    return result[train_columns]

pseudo_strong = make_train_format(dev_strong)
pseudo_all = make_train_format(high_conf_dev)

print("강한 신뢰 데이터:", pseudo_strong.shape)
print("전체 후보 데이터:", pseudo_all.shape)

In [ ]:
from google.colab import drive
import os

drive.mount("/content/drive")

SAVE_DIR = "/content/drive/MyDrive/ssafy/ai-challenge"
os.makedirs(SAVE_DIR, exist_ok=True)

pseudo_strong.to_csv(
    f"{SAVE_DIR}/dev_pseudo_strong_267.csv",
    index=False,
    encoding="utf-8-sig"
)

pseudo_all.to_csv(
    f"{SAVE_DIR}/dev_pseudo_all_2026.csv",
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료:", SAVE_DIR)

In [ ]:
from sklearn.model_selection import train_test_split

SEED = 43

train_split, valid_split = train_test_split(
    train,
    test_size=0.1,
    random_state=SEED,
    stratify=train["answer"]
)

train_split = train_split.reset_index(drop=True)
valid_split = valid_split.reset_index(drop=True)

print("학습 데이터:", train_split.shape)
print("검증 데이터:", valid_split.shape)

print("\n검증 정답 분포")
print(valid_split["answer"].value_counts().sort_index())

In [ ]:
import os

SPLIT_DIR = "/content/drive/MyDrive/ssafy/ai-challenge/split"
os.makedirs(SPLIT_DIR, exist_ok=True)

train_split.to_csv(
    f"{SPLIT_DIR}/train_split_seed43.csv",
    index=False,
    encoding="utf-8-sig"
)

valid_split.to_csv(
    f"{SPLIT_DIR}/valid_split_seed43.csv",
    index=False,
    encoding="utf-8-sig"
)

train_split[["id"]].to_csv(
    f"{SPLIT_DIR}/train_ids_seed43.csv",
    index=False
)

valid_split[["id"]].to_csv(
    f"{SPLIT_DIR}/valid_ids_seed43.csv",
    index=False
)

print("Seed 43 분할 저장 완료")

In [ ]:
exp_base = train_split.copy()

exp_strong = pd.concat(
    [train_split, pseudo_strong],
    ignore_index=True
)

exp_all = pd.concat(
    [train_split, pseudo_all],
    ignore_index=True
)

print("Base:", exp_base.shape)
print("Base + Strong Dev:", exp_strong.shape)
print("Base + All Dev:", exp_all.shape)

In [ ]:
EXP_DIR = "/content/drive/MyDrive/ssafy/ai-challenge/experiment_data"
os.makedirs(EXP_DIR, exist_ok=True)

exp_base.to_csv(
    f"{EXP_DIR}/exp01_train_only_seed43.csv",
    index=False,
    encoding="utf-8-sig"
)

exp_strong.to_csv(
    f"{EXP_DIR}/exp02_train_plus_strong_seed43.csv",
    index=False,
    encoding="utf-8-sig"
)

exp_all.to_csv(
    f"{EXP_DIR}/exp03_train_plus_all_seed43.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Seed 43 실험 데이터 저장 완료")

In [ ]:
# 질문과 선택지가 완전히 동일한 데이터를 찾기 위한 키
key_cols = ["question", "a", "b", "c", "d"]

train_keys = train[key_cols].astype(str).agg(" || ".join, axis=1)
dev_keys = dev[key_cols].astype(str).agg(" || ".join, axis=1)
test_keys = test[key_cols].astype(str).agg(" || ".join, axis=1)

print("===== 데이터 내부 중복 =====")
print("Train 내부 중복:", train_keys.duplicated().sum())
print("Dev 내부 중복:", dev_keys.duplicated().sum())
print("Test 내부 중복:", test_keys.duplicated().sum())

print("\n===== 데이터셋 간 동일 문제 =====")
print("Train-Dev 중복:", len(set(train_keys) & set(dev_keys)))
print("Train-Test 중복:", len(set(train_keys) & set(test_keys)))
print("Dev-Test 중복:", len(set(dev_keys) & set(test_keys)))

In [ ]:
print("===== ID 중복 =====")
print("Train ID 내부 중복:", train["id"].duplicated().sum())
print("Dev ID 내부 중복:", dev["id"].duplicated().sum())
print("Test ID 내부 중복:", test["id"].duplicated().sum())

print("\n===== 경로 중복 =====")
print("Train 경로 내부 중복:", train["path"].duplicated().sum())
print("Dev 경로 내부 중복:", dev["path"].duplicated().sum())
print("Test 경로 내부 중복:", test["path"].duplicated().sum())

In [ ]:
common_keys = set(train_keys) & set(test_keys)

for key in common_keys:
    train_idx = train_keys[train_keys == key].index
    test_idx = test_keys[test_keys == key].index

    print("===== Train 중복 후보 =====")
    display(train.loc[train_idx])

    print("===== Test 중복 후보 =====")
    display(test.loc[test_idx])

In [ ]:
def classify_question(question):
    q = str(question)

    if any(word in q for word in ["전화번호", "연락처"]):
        return "전화번호"

    if any(word in q for word in [
        "가격", "금액", "얼마", "원인가", "원으로"
    ]):
        return "가격·금액"

    if any(word in q for word in [
        "날짜", "연도", "몇 년", "몇 월", "몇 일",
        "시간", "몇 시"
    ]):
        return "날짜·시간"

    if any(word in q for word in [
        "몇 개", "몇 명", "몇 번째", "개수", "수량"
    ]):
        return "수량·숫자"

    if any(word in q for word in [
        "어디", "위치", "방향", "장소", "지역"
    ]):
        return "위치·방향"

    if any(word in q for word in [
        "이름", "상호명", "명칭", "브랜드", "제품명"
    ]):
        return "이름·명칭"

    if any(word in q for word in [
        "문구", "적힌", "쓰여", "표시", "글자"
    ]):
        return "문자·OCR"

    return "기타"

train["question_type"] = train["question"].apply(classify_question)
dev["question_type"] = dev["question"].apply(classify_question)
test["question_type"] = test["question"].apply(classify_question)

for name, df in {
    "Train": train,
    "Dev": dev,
    "Test": test
}.items():
    print(f"\n===== {name} 질문 유형 =====")
    print(df["question_type"].value_counts())
    print(
        df["question_type"]
        .value_counts(normalize=True)
        .mul(100)
        .round(2)
    )

In [ ]:
train_split["question_type"] = train_split["question"].apply(classify_question)
valid_split["question_type"] = valid_split["question"].apply(classify_question)

print("===== 학습 질문 유형 비율 =====")
print(
    train_split["question_type"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\n===== 검증 질문 유형 비율 =====")
print(
    valid_split["question_type"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)